# Run all notebooks in the project

In [1]:
# Header for the notebook
from datetime import datetime
from IPython.display import display, Markdown

# Get the current date
title = "Movement smoothness as a subclinical marker in low back pain - Segmentation and SPARC/NNP calculation"
current_date = datetime.now().strftime("%d %B %Y, %H:%M:%S")
authors = "Ancelin Gely (and Copilot)"

# Insert the date into the notebook
display(Markdown(f"# {title}"))
display(Markdown(f"{current_date}"))
display(Markdown(f"by {authors}"))

# Movement smoothness as a subclinical marker in low back pain - Segmentation and SPARC/NNP calculation

07 May 2026, 09:12:49

by Ancelin Gely (and Copilot)

# Python Project : Load, filter and analyze gyroscope data to compute SPARC and NNP for trunck flexion

## Aim of the code 

The aim of this code is to extract the velocity profile from the gyroscope data, segment the movement into repetition of flexion - return from flexion (extension) and compute the SPARC and NNP for each repetition and for the whole movement. 

## Data organization

### File name 
Flexion_Dos_OOX_av.xlsx
X is the patient ID (1-10)

### File structure
The data is organized as a table with different sheets : 
- general informations 
- Markers
- Segment orientation - Quat
- Segment orientation - Euler
- Segment position 
- Segment velocity
- Segment acceleration
- Segment angular velocity
- Ergonomic joint angle 
- Center of mass
- Sensor free acceleration
- Sensor magnetic field
- Sensor orientation - Quat
- Sensor orientation - Euler

### Gyroscope data
The gyroscope data is located in the sheet "Segment angular velocity". It contains the angular velocity of the segment in three dimensions (x, y, z). 

Columns :
- Frame	
- Pelvis x, y, z		
- L5 x, y, z
- L3 x, y, z
- T12 x, y, z	
- T8 x, y, z

## Notebook organization

In this jupiter notebook, we will perform the following steps :
1. Load the data and filter the gyroscope data for L3 Y (L3)
2. Segment the movement into repetitions of flexion return from flexion (extension)
3. Compute the SPARC and NNP for each repetition and for the whole movement
4. Add the results to a dataframe and save it as a xlsx file


### Compute SPARC and NNP 

#### Spectral arc lenght (SPARC) computation : 

The spectral arc lenght (SPARC) is a frequency-based metric that quantifies the smoothness of a movement by analyzing the Fourier magnitude spectrum of the velocity profile. It is calculated using the following formula:

$SPARC = - ∫ |dV(ω)/dω| dω$ 

Where V(ω) is the Fourier magnitude spectrum of the velocity profile and ω is the angular frequency. 

The SPARC corresponds to the negative integral of the absolute value of the derivative of the Fourier magnitude spectrum with respect to frequency.

SPARC values typically range from 0 to -∞, with higher values indicating smoother movements. A SPARC value of 0 indicates a perfectly smooth movement, while a SPARC value of -∞ indicates a highly irregular movement.

#### Number of peaks (NNP) computation :

The number of peaks (NNP) is a time-domain metric that quantifies the smoothness of a movement by counting the number of peaks in the velocity profile. The result is then normalized by the movement duration to account for differences in movement speed. It is calculated using the following formula:

$NNP = \frac{\text{number of peaks}}{\text{movement duration}}$

NNP values typically range from 0 to +∞, with lower values indicating smoother movements. A NNP value of 0 indicates a perfectly smooth movement, while higher NNP values indicate more irregular movements with more peaks in the velocity profile.


# Code 

## Import libraries



In [2]:
# If you need to install the libraries, uncomment the following lines and run them once. After that, you can comment them again.
# !pip install numpy
# !pip install pandas
# !pip install matplotlib
# !pip install scipy
# !pip install IPython

# Import library
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from scipy.signal import butter
from scipy.signal import filtfilt
from scipy.signal import find_peaks

## Function definitions 

The 4 steps (load and filter the gyroscope data, segment the movement, compute the SPARC and NNP, and save the results) of the code are defined as functions "analyse_signal" to make the code more organized and reusable.

It uses the following functions :
- butterworth_filter : to filter the signal with a butterworth filter
- find_peaks : to find the peaks in the velocity profile
- compute_SPARC : to compute the SPARC of the signal
- fft : to compute the Fourier transform of the signal




### Load the data and filter the gyroscope data for L3 Y (L3)

#### Load and reshape the data
The data is loaded from the excel file using pandas and the gyroscope data for L3 Y is extracted. Data are in frame so the time is calculated by dividing the frame number by the sampling frequency (60 Hz).

$time = \frac{frame}{\text{sampling frequency}}$

#### Filter the signal
A low pass Butterworth filter is applied to the data to obtain a smoother velocity profile. 

The filter is composed of (copilote): 

- order : 2 = determines the steepness of the filter. An order of 2 provides a moderate roll-off, which is suitable for removing high-frequency noise while preserving the main characteristics of the movement signal.

- cutoff frequency : 10 Hz = determines the frequency at which the filter starts to attenuate the signal. A cutoff frequency of 10 Hz is appropriate for human movement analysis, as it allows to retain the relevant movement information while filtering out high-frequency noise.

- fc/(fs/2) : the cutoff frequency is normalized by the Nyquist frequency (half of the sampling frequency) to ensure that the filter is designed correctly for the given sampling rate.

- b, a : the filter coefficients are calculated using the Butterworth filter design function from the scipy library. These coefficients are then used to apply the filter to the signal using the filtfilt function, which applies the filter forward and backward to avoid phase distortion.

- low pass : the type of filter is specified as low pass, which means that it allows frequencies below the cutoff frequency to pass through while attenuating frequencies above the cutoff frequency. 

In [3]:
#######################################
######## LOAD + FILTER ################
#######################################

def load_and_filter_signal(file_path, sheet_name, column_name, fs, fc):

    signal = pd.read_excel(file_path, sheet_name=sheet_name)

    if "Time (s)" not in signal.columns:

        if "Frame" not in signal.columns:
            raise ValueError("Time (s) or Frame column missing")

        signal["Time (s)"] = signal["Frame"] / fs

    omega = signal[column_name].to_numpy()

    # Low-pass filter
    b, a = butter(2, fc / (fs / 2), btype="low")

    omega_filt = filtfilt(b, a, omega)

    return signal, omega_filt

### Segment detection
Patients were asked to perform 5 repetitions of flexion-extension, so the signal is composed of 5 segments of flexion and 5 segments of extension.

The movement is segmented into repetitions of flexion-extension by transforming the velocity profile into a binary signal using a threshold value. The threshold is set to 0.01 to ensure that only significant movements are detected, while small fluctuations in the signal are ignored. 

The binary signal is created as follows :

- 1 : Values above the threshold
- 0 : Values below the threshold

Then, the difference between the binary signal is calculated to identify the start and end of each repetition. The difference is computed as follows :

- 1 (1-0) : transition from extension to flexion, indicating the start of a flexion phase
- -1 (0-1) : transition from flexion to extension, indicating the start of an extension phase
- 0 : no change in the movement phase

The start and end of each repetition are then associated with peaks in the velocity profile to ensure that the segmentation corresponds to actual movement phases. The find_peaks function from the scipy library is used to identify the peaks in the velocity profile, which correspond to the maximum velocity during each flexion and extension phase.

Segments shorter than 0.5 seconds are excluded from the analysis to ensure that only complete repetitions are considered. This is done by calculating the duration of each segment and filtering out those that do not meet the minimum duration criterion.

Consecutive segments of the same type (flexion or extension) are suppressed to avoid counting artifacts. This is achieved by checking for consecutive segments of the same type and keeping only the first segment in such cases.


In [4]:
#######################################
######## SEGMENT DETECTION ############
#######################################

def detect_segments(omega_filt, eps, fs, min_peak_distance, signal):

    # Negative segments
    neg_mask = omega_filt < -eps

    neg_transitions = np.diff(neg_mask.astype(int))

    neg_starts = np.where(neg_transitions == 1)[0] + 1
    neg_ends = np.where(neg_transitions == -1)[0] + 1

    if neg_mask[0]:
        neg_starts = np.insert(neg_starts, 0, 0)

    if neg_mask[-1]:
        neg_ends = np.append(neg_ends, len(neg_mask))

    # Positive segments
    pos_mask = omega_filt > eps

    pos_transitions = np.diff(pos_mask.astype(int))

    pos_starts = np.where(pos_transitions == 1)[0] + 1
    pos_ends = np.where(pos_transitions == -1)[0] + 1

    if pos_mask[0]:
        pos_starts = np.insert(pos_starts, 0, 0)

    if pos_mask[-1]:
        pos_ends = np.append(pos_ends, len(pos_mask))

    # Peak detection
    positive_peaks, _ = find_peaks(
        omega_filt,
        prominence=0.9 * np.std(omega_filt),
        distance=int(min_peak_distance * fs)
    )

    negative_peaks, _ = find_peaks(
        -omega_filt,
        prominence=0.9 * np.std(omega_filt),
        distance=int(min_peak_distance * fs)
    )

    # Associate peaks with segments
    negative_segments = []

    for p in negative_peaks:

        idx = np.where((neg_starts <= p) & (neg_ends >= p))[0]

        if len(idx) == 1:

            negative_segments.append(
                (int(neg_starts[idx[0]]), int(neg_ends[idx[0]]))
            )

    positive_segments = []

    for p in positive_peaks:

        idx = np.where((pos_starts <= p) & (pos_ends >= p))[0]

        if len(idx) == 1:

            positive_segments.append(
                (int(pos_starts[idx[0]]), int(pos_ends[idx[0]]))
            )

    # Remove segments shorter than 0.5 s
    negative_segments = [
        (s, e) for s, e in negative_segments
        if (signal["Time (s)"].iloc[e - 1] - signal["Time (s)"].iloc[s]) >= 0.5
    ]

    positive_segments = [
        (s, e) for s, e in positive_segments
        if (signal["Time (s)"].iloc[e - 1] - signal["Time (s)"].iloc[s]) >= 0.5
    ]

    # Suppress consecutive segments
    def suppress_consecutive_segments(segments):

        if not segments:
            return []

        suppressed = [segments[0]]

        for s, e in segments[1:]:

            last_s, last_e = suppressed[-1]

            if (s - last_e) > 1:
                suppressed.append((s, e))

        return suppressed

    negative_segments = suppress_consecutive_segments(negative_segments)
    positive_segments = suppress_consecutive_segments(positive_segments)

    return positive_segments, negative_segments

### SPARC computation
The SPARC is computed using the following steps (copilote):

1. center the velocity profile by subtracting the mean from the signal to ensure that the Fourier transform captures the relevant frequency components of the movement.

2. Compute the Fourier transform of the velocity profile using the fft function from the scipy library to obtain the Fourier magnitude spectrum.

3. Normalize the Fourier magnitude spectrum by dividing it by its maximum value to ensure that the SPARC value is independent of the signal amplitude.  

4. apply a log transformation to the normalized Fourier magnitude spectrum to reduce the influence of large values and enhance the contribution of smaller values, which can provide a more accurate representation of the movement smoothness.

5. Calculate the derivative of the Fourier magnitude spectrum with respect to frequency.

6. compute the frequency range over which the SPARC is calculated. 

7. Compute the absolute value of the derivative and root square it over the frequency range to obtain the SPARC value.

8. The SPARC value is then negated to ensure that higher values correspond to smoother movements, as the original formula yields negative values for smoother movements.

In [5]:
#######################################
############ COMPUTE SPARC ############
#######################################

def compute_sparc(x, fs):

    x = x - np.mean(x)

    n = len(x)

    spectrum = np.abs(np.fft.rfft(x))

    max_spectrum = np.max(spectrum)

    if max_spectrum == 0:
        return np.nan

    spectrum = spectrum / max_spectrum

    spectrum += 1e-10

    freqs = np.fft.rfftfreq(n, d=1/fs)

    threshold = 0.05

    valid = spectrum > threshold

    spectrum = spectrum[valid]
    freqs = freqs[valid]

    log_spectrum = np.log(spectrum)

    df = np.diff(freqs)
    ds = np.diff(log_spectrum)

    arc_length = np.sum(np.sqrt(df**2 + ds**2))

    return -arc_length

In [6]:
def analyze_signal_v2(file_path, sheet_name, column_name):

    # Parameters
    fs = 60
    fc = 10
    eps = 0.01
    min_peak_distance = 0.5

    ###################################
    ######## LOAD + FILTER ############
    ###################################

    signal, omega_filt = load_and_filter_signal(
        file_path,
        sheet_name,
        column_name,
        fs,
        fc
    )

    ###################################
    ######## SEGMENTATION #############
    ###################################

    positive_segments, negative_segments = detect_segments(
        omega_filt,
        eps,
        fs,
        min_peak_distance,
        signal
    )

    ###################################
    ########### SPARC #################
    ###################################

    positive_sparc = []

    for start, end in positive_segments:

        positive_sparc.append(
            compute_sparc(omega_filt[start:end], fs)
        )

    negative_sparc = []

    for start, end in negative_segments:

        negative_sparc.append(
            compute_sparc(omega_filt[start:end], fs)
        )

    SPARC = compute_sparc(omega_filt, fs)

    ###################################
    ############ NNP ##################
    ###################################

    extension_peak_counts = []

    for s, e in negative_segments:

        segment_time = signal["Time (s)"].iloc[s:e]

        segment_omega = omega_filt[s:e]

        peaks, _ = find_peaks(
            segment_omega,
            prominence=0.01
        )

        peak_count = len(peaks)

        duration = (
            segment_time.iloc[-1]
            - segment_time.iloc[0]
        )

        normalized_peak_count = (
            peak_count / duration
            if duration > 0 else np.nan
        )

        extension_peak_counts.append(
            normalized_peak_count
        )

    flexion_peak_counts = []

    for s, e in positive_segments:

        segment_time = signal["Time (s)"].iloc[s:e]

        segment_omega = omega_filt[s:e]

        peaks, _ = find_peaks(
            segment_omega,
            prominence=0.01
        )

        peak_count = len(peaks)

        duration = (
            segment_time.iloc[-1]
            - segment_time.iloc[0]
        )

        normalized_peak_count = (
            peak_count / duration
            if duration > 0 else np.nan
        )

        flexion_peak_counts.append(
            normalized_peak_count
        )

    ###################################
    ######## TOTAL SIGNAL NNP #########
    ###################################

    peaks, _ = find_peaks(
        omega_filt,
        prominence=0.005
    )

    total_duration = (
        signal["Time (s)"].iloc[-1]
        - signal["Time (s)"].iloc[0]
    )

    nnp = (
        len(peaks) / total_duration
        if total_duration > 0 else np.nan
    )

    ###################################
    ######## RESULTS TABLE ############
    ###################################

    signal_name = file_path.split("/")[-1].split(".")[0]

    results = pd.DataFrame({

        "Signal name": [signal_name],

        "SPARC": [SPARC],

        "NNP": [nnp],

        "mean_SPARC_flexion": [
            np.nanmean(positive_sparc)
        ],

        "mean_NNP_flexion": [
            np.nanmean(flexion_peak_counts)
        ]
    })

    return results

In [7]:
# Create a loop to analyze all the signals and concatenate the results in a single dataframe.
# be carefull with the 10th file because it is named "FlexionDos_010_av.xlsx" and not "FlexionDos_0010_av.xlsx".

results_all = pd.DataFrame()
for i in range(1, 11):

    file_number = f"{i:03d}"

    file_path = f"data/FlexionDos_{file_number}_av.xlsx"

    sheet_name = "Segment Angular Velocity"

    column_name = "L3 y"

    result = analyze_signal_v2(file_path, sheet_name, column_name)

    results_all = pd.concat([results_all, result], ignore_index=True)

In [8]:
results_all

,Signal name,SPARC,NNP,mean_SPARC_flexion,mean_NNP_flexion
0,FlexionDos_001_av,-6.874907,1.670103,-4.159162,1.275352
1,FlexionDos_002_av,-6.735183,3.946588,-5.349642,3.040954
2,FlexionDos_003_av,-11.533392,3.625192,-5.434564,4.058155
3,FlexionDos_004_av,-8.011311,1.224490,-2.997148,1.109087
4,FlexionDos_005_av,-12.947031,5.095796,-15.475841,4.736775
5,FlexionDos_006_av,-10.508508,2.743022,-3.215461,2.022103
6,FlexionDos_007_av,-7.602682,1.015572,-3.276752,0.800853
7,FlexionDos_008_av,-9.216218,3.790782,-6.385948,3.430441
8,FlexionDos_009_av,-8.477055,1.893048,-2.691233,1.888556
9,FlexionDos_010_av,-9.464665,4.101877,-6.167191,3.932310


In [ ]:
# Save as xlsx in the folder "results"
results_all.to_excel("results/results_all.xlsx", index=False)
